In [2]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import xml.etree.ElementTree as ET
from tqdm.notebook import tqdm

# --- UPDATED CONFIGURATION ---
CONFIG = {
    'xml_root': r'H:\DPJI\IDDPedestrian\annotations\gopro',
    'img_root': r'H:\DPJI\fast_data_static_v2', 
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'obs_len': 15,  
    'pred_len': 45, 
    'hidden_size': 256,
    'embed_size': 128, 
    'batch_size': 64,  
    'epochs': 30,      
    'lr': 5e-4
}

print(f"✅ Libraries Imported. Device: {CONFIG['device']}")

✅ Libraries Imported. Device: cuda


In [3]:
class IDDTrajectoryDataset(Dataset):
    def __init__(self, xml_root, obs_len=15, pred_len=45):
        self.obs_len = obs_len
        self.pred_len = pred_len
        self.seq_len = obs_len + pred_len
        self.samples = []
        
        xml_files = glob.glob(os.path.join(xml_root, '**', '*.xml'), recursive=True)
        for xml in tqdm(xml_files, desc="Parsing XMLs"):
            try:
                tree = ET.parse(xml)
                root = tree.getroot()
                for track in root.findall('track'):
                    if track.attrib['label'] != 'pedestrian': continue
                    
                    track_data = []
                    for box in track.findall('box'):
                        xtl, ytl = float(box.attrib['xtl']), float(box.attrib['ytl'])
                        xbr, ybr = float(box.attrib['xbr']), float(box.attrib['ybr'])
                        track_data.append([(xtl+xbr)/2, (ytl+ybr)/2, xbr-xtl, ybr-ytl])
                    
                    track_data = np.array(track_data)
                    if len(track_data) < self.seq_len: continue

                    for i in range(0, len(track_data) - self.seq_len + 1, 5):
                        full_seq = track_data[i : i + self.seq_len]
                        
                        # Calculate Displacements (Velocity)
                        # displacement_t = pos_t - pos_{t-1}
                        # We pad the first observation with 0 or repeat
                        disp = np.zeros_like(full_seq[:, :2])
                        disp[1:] = full_seq[1:, :2] - full_seq[:-1, :2]
                        
                        self.samples.append({
                            'obs_pos': full_seq[:obs_len, :2],      # Absolute pos for reference
                            'obs_disp': disp[:obs_len],             # Input: Displacements
                            'pred_disp': disp[obs_len:],            # Target: Displacements
                            'pred_abs': full_seq[obs_len:],         # Target: Absolute (for loss/metrics)
                            'last_obs_pos': full_seq[obs_len-1, :2], # Pivot point
                            'gt_wh': full_seq[obs_len:, 2:]         # W, H for CF-MSE
                        })
            except: pass

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return (
            torch.tensor(s['obs_disp'], dtype=torch.float32),
            torch.tensor(s['pred_disp'], dtype=torch.float32),
            torch.tensor(s['last_obs_pos'], dtype=torch.float32),
            torch.tensor(s['pred_abs'], dtype=torch.float32),
            torch.tensor(s['gt_wh'], dtype=torch.float32)
        )

dataset = IDDTrajectoryDataset(CONFIG['xml_root'])
print(f"✅ Dataset Created: {len(dataset)} sequences.")

Parsing XMLs:   0%|          | 0/33 [00:00<?, ?it/s]

✅ Dataset Created: 52858 sequences.


In [4]:
class PIETrajNet(nn.Module):
    def __init__(self, input_size=2, hidden_size=256, embed_size=128):
        super(PIETrajNet, self).__init__()
        self.embed = nn.Sequential(nn.Linear(input_size, embed_size), nn.ReLU())
        self.encoder = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.decoder = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 2)
        
    def forward(self, obs_disp, pred_len):
        batch_size = obs_disp.size(0)
        emb = self.embed(obs_disp)
        _, (h, c) = self.encoder(emb)
        
        # Start decoding with the last observed displacement
        curr_disp = obs_disp[:, -1, :].unsqueeze(1)
        predictions = []
        
        for _ in range(pred_len):
            curr_emb = self.embed(curr_disp)
            out, (h, c) = self.decoder(curr_emb, (h, c))
            disp_pred = self.fc(out)
            predictions.append(disp_pred)
            curr_disp = disp_pred # Autoregressive loop
            
        return torch.cat(predictions, dim=1)

model = PIETrajNet(hidden_size=CONFIG['hidden_size'], embed_size=CONFIG['embed_size']).to(CONFIG['device'])

In [7]:
# --- REVISED TRAINING CELL ---
from torch.utils.data import random_split

# Calculate lengths for split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_set, batch_size=CONFIG['batch_size'], shuffle=False)

optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])
criterion = nn.MSELoss()

print(f"🚀 Starting Training on {len(train_set)} samples...")

for epoch in range(CONFIG['epochs']):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}", leave=False)
    
    for disp_in, disp_gt, last_pos, abs_gt, _ in pbar:
        disp_in = disp_in.to(CONFIG['device'])
        last_pos = last_pos.to(CONFIG['device'])
        abs_gt = abs_gt.to(CONFIG['device'])
        
        optimizer.zero_grad()
        
        # 1. Forward pass: Predict displacements [Batch, 45, 2]
        pred_disps = model(disp_in, CONFIG['pred_len'])
        
        # 2. Reconstruct absolute positions [Batch, 45, 2]
        pred_abs = last_pos.unsqueeze(1) + torch.cumsum(pred_disps, dim=1)
        
        # 3. FIX: Slice abs_gt to [Batch, 45, 2] to match pred_abs
        # abs_gt is [Batch, 45, 4] -> take only cx, cy
        target_pos = abs_gt[:, :, :2]
        
        # 4. Calculate loss
        loss = criterion(pred_abs, target_pos)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.2f}"})
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for disp_in, _, last_pos, abs_gt, _ in val_loader:
            disp_in = disp_in.to(CONFIG['device'])
            last_pos = last_pos.to(CONFIG['device'])
            abs_gt = abs_gt.to(CONFIG['device'])
            
            pred_disps = model(disp_in, CONFIG['pred_len'])
            pred_abs = last_pos.unsqueeze(1) + torch.cumsum(pred_disps, dim=1)
            
            # Match sizes for validation loss too
            target_pos = abs_gt[:, :, :2]
            val_loss += criterion(pred_abs, target_pos).item()
            
    print(f"Epoch {epoch+1} | Train Loss: {total_loss/len(train_loader):.2f} | Val Loss: {val_loss/len(val_loader):.2f}")

torch.save(model.state_dict(), "pietraj_velocity_model.pth")
print("💾 Model Saved.")

🚀 Starting Training on 42286 samples...


Epoch 1/30:   0%|          | 0/661 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 2345.79 | Val Loss: 1989.24


Epoch 2/30:   0%|          | 0/661 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 1971.53 | Val Loss: 1954.74


...

Epoch 28 | Train Loss: 1409.35 | Val Loss: 1816.33


Epoch 29/30:   0%|          | 0/661 [00:00<?, ?it/s]

Epoch 29 | Train Loss: 1379.39 | Val Loss: 1839.24


Epoch 30/30:   0%|          | 0/661 [00:00<?, ?it/s]

Epoch 30 | Train Loss: 1368.67 | Val Loss: 1864.95
💾 Model Saved.


In [9]:
def evaluate_pietraj(model, loader):
    model.eval()
    all_mse, all_cmse, all_cfmse = [], [], []
    
    with torch.no_grad():
        for disp_in, _, last_pos, abs_gt, wh_gt in loader:
            disp_in = disp_in.to(CONFIG['device'])
            
            # 1. Get predictions [Batch, 45, 2]
            pred_disps = model(disp_in, CONFIG['pred_len']).cpu().numpy()
            
            # Convert tensors to numpy
            abs_gt = abs_gt.numpy()    # [Batch, 45, 4] -> (cx, cy, w, h)
            wh_gt = wh_gt.numpy()      # [Batch, 45, 2] -> (w, h)
            last_pos = last_pos.numpy() # [Batch, 2]
            
            # 2. Reconstruct Absolute Path [Batch, 45, 2]
            pred_abs = last_pos[:, np.newaxis, :] + np.cumsum(pred_disps, axis=1)
            
            # 3. MSE (Average Squared Error over the 45 frames)
            # FIX: Use abs_gt[:, :, :2] to match pred_abs dimensions
            sq_diff = np.sum((pred_abs - abs_gt[:, :, :2])**2, axis=2) 
            all_mse.extend(np.mean(sq_diff, axis=1))
            
            # 4. C-MSE (Center Error at the final frame / 1.5s)
            # FIX: Use abs_gt[:, -1, :2]
            c_mse = np.sum((pred_abs[:, -1, :] - abs_gt[:, -1, :2])**2, axis=1)
            all_cmse.extend(c_mse)
            
            # 5. CF-MSE (Center + Foot Error at the final frame)
            # Foot Y = Center Y + (Height / 2)
            # We use the ground truth height (wh_gt) as per standard PIETraj evaluation
            gt_foot_y = abs_gt[:, -1, 1] + (wh_gt[:, -1, 1] / 2.0)
            pred_foot_y = pred_abs[:, -1, 1] + (wh_gt[:, -1, 1] / 2.0)
            
            # Foot MSE = (Pred_Cx - GT_Cx)^2 + (Pred_FootY - GT_FootY)^2
            f_mse = (pred_abs[:, -1, 0] - abs_gt[:, -1, 0])**2 + (pred_foot_y - gt_foot_y)**2
            all_cfmse.extend(c_mse + f_mse)

    return np.mean(all_mse), np.mean(all_cmse), np.mean(all_cfmse)

# Run the evaluation
mse, cmse, cfmse = evaluate_pietraj(model, val_loader)

print("-" * 30)
print(f"📊 FINAL PIETRAJ RESULTS (IDD-PeD):")
print(f"MSE (Avg):   {mse:.2f}  (Table III baseline: 2181)")
print(f"C-MSE:       {cmse:.2f}  (Table III baseline: 1979)")
print(f"CF-MSE:      {cfmse:.2f}  (Table III baseline: 8960)")
print("-" * 30)

------------------------------
📊 FINAL PIETRAJ RESULTS (IDD-PeD):
MSE (Avg):   3737.19  (Table III baseline: 2181)
C-MSE:       16463.08  (Table III baseline: 1979)
CF-MSE:      32926.17  (Table III baseline: 8960)
------------------------------
